# **Clasificación de Sentimientos de Reseñas de Películas en IMDb con Redes Neuronales Recurrentes**

Los ratings y reseñas de películas son una fuente rica de datos textuales que pueden ser analizados para identificar el sentimiento subyacente, ya sea positivo o negativo. El uso de redes neuronales recurrentes (RNN) ha demostrado ser eficaz en la clasificación de secuencias de texto, ya que estas redes pueden capturar la información contextual en oraciones y palabras, identificando patrones relevantes para determinar la polaridad de una reseña.

**Redes Neuronales Recurrentes**


Las RNN son especialmente adecuadas para el análisis de datos secuenciales como texto, ya que tienen la capacidad de almacenar y recordar información previa en la secuencia, lo cual es crucial para entender el contexto. Al aplicar una RNN a reseñas de películas, la red puede ser entrenada para identificar palabras, frases y patrones que generalmente se asocian con sentimientos positivos o negativos.

**Procesamiento de Texto**

El procesamiento de texto para este proyecto implica convertir las reseñas en un formato numérico que la RNN pueda procesar. Esto se logra a través de técnicas como la tokenización y la vectorización de palabras (por ejemplo, utilizando embeddings). Luego, la RNN analizará la secuencia de palabras para clasificar cada reseña en una de dos categorías: positiva o negativa.

**Ventajas y Aplicaciones**

Implementar RNN para la clasificación de sentimientos en reseñas de películas ofrece varias ventajas:
- Detección precisa de emociones: Mejora la capacidad de capturar sentimientos más matizados en textos largos y complejos.
- Adaptabilidad a distintos idiomas y contextos: Las RNN pueden ser entrenadas en diferentes conjuntos de datos para adaptarse a lenguajes y contextos específicos, como géneros de películas o plataformas de reseñas.
- Asistencia en sistemas de recomendación: Permite mejorar las recomendaciones personalizadas en plataformas de streaming basadas en el análisis de reseñas de usuarios.

**A. Objetivo**
- Desarrollar un método basado en redes neuronales recurrentes que permita clasificar con precisión las reseñas de películas en IMDb en una de las dos categorías: positiva o negativa.

**B. Conjunto de datos**
- Los datos corresponden a un dataset de reseñas de películas en IMDb, disponibles en el siguiente [enlace](https://www.kaggle.com/datasets/yasserh/imdb-movie-ratings-sentiment-analysis/data). El dataset contiene miles de reseñas etiquetadas con su correspondiente rating y etiqueta de sentimiento (positivo o negativo).

**C. Actividades por realizar**
1. Preprocesamiento de las reseñas: Construir un pipeline que permita limpiar y tokenizar las reseñas de texto, convirtiéndolas en secuencias numéricas utilizando embeddings (por ejemplo, Word2Vec o GloVe).
2. Desarrollo de la arquitectura de red neuronal recurrente: Implementar una RNN (por ejemplo, LSTM o GRU) para la clasificación de las reseñas en dos categorías. La arquitectura de la red es de libre elección, pero se sugiere incluir capas de embedding, capas recurrentes y una capa densa de salida con activación sigmoide. Justificar la elección de cada capa del modelo.
3. Entrenamiento y evaluación del modelo: Dividir el dataset en conjuntos de entrenamiento, validación y prueba. Entrenar el modelo utilizando el conjunto de entrenamiento y evaluar su rendimiento con el conjunto de prueba, reportando métricas de precisión, recall y F1-score.

## Importar librerias

In [ ]:
!pip install gensim

In [ ]:
pip install nltk

In [ ]:
pip install contractions

In [ ]:
import pandas as pd               # Librería para manejo de dataframes
import numpy as np                # Librería para trabajar con datos matriciales
import matplotlib.pyplot as plt   # Librería de visualización
import nltk
import contractions as ct
import re
import string
import os
import urllib.request
import zipfile

nltk.download('wordnet')
nltk.download('omw-1.4')  # Recomendado para soporte multilingüe/actualizado de wordnet

#Preprocesamiento de texto
nltk.download('stopwords')
from nltk import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.decomposition import TruncatedSVD

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score   # Importar métricas de validación
from sklearn.preprocessing import StandardScaler      # Importar objeto para realizar estandarización de variables
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score

from time import time   # Librería para cálculos de tiempos de ejecución
import copy              # Librería para copiar los mejores pesos del modelo durante el entrenamiento




# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader   # Utilidades para construir el pipeline de datos

from gensim.models import Word2Vec


## Inicialización de semillas.

In [ ]:
SEED = 0

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    return seed


SEED = seed_everything(42)

## Uso de dispositivo de computo.

In [ ]:
# Dispositivo de cómputo (usa GPU si está disponible de lo contrario CPU)
if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

## **Carga de datos**

Realizaremos la carga de datos usando la función de Pandas read_csv(), especificando la ruta y el separador del archivo.

In [ ]:
data = pd.read_csv('movie.csv', sep=',')
data.info()

**Verificación de etiquetas distintas en dataset**

In [ ]:
clases_unicas = data['label'].unique()
print(f"Etiquetas únicas: {clases_unicas}")

De lo anterior se observa que hay dos clases de sentimientos en el dataset:

0: Sentimiento positivo (según fuente del dataset)

1: Sentimiento negativo (según fuente del dataset)

## Tratamiento de datos nulos y duplicados:

In [ ]:
# Datos nulos
print(f'Data:\n{data.isna().sum()}')

In [ ]:
print("\nDistribución de etiquetas de polaridad (0: Negativo, 1: Positivo):")
conteo_clases = data['label'].value_counts()
print(conteo_clases)

# Visualización gráfica de la distribución de clases
plt.figure(figsize=(13, 4))
barras = plt.bar(['Negativo (0)', 'Positivo (1)'], conteo_clases.values, color=['#5a2ff4', '#60d13e'], width=0.4)
plt.title('Distribución de clases en el dataset IMDb (movie.csv)', fontsize=12)
plt.xlabel('Sentimiento', fontsize=11)
plt.ylabel('Cantidad de reseñas', fontsize=11)
plt.grid(axis='y', linestyle=':', alpha=0.7)

for barra in barras:
    yval = barra.get_height()
    plt.text(barra.get_x() + barra.get_width()/2, yval + 300, f"{yval:,}", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

De lo anterior no hay datos nulos en el dataset.

In [ ]:
# Duplicados
data.duplicated().sum()

In [ ]:
data[data.duplicated(keep=False)].sort_values(by='text').head(10)

De lo anterior se observa que hay 277 registros duplicados en el dataset de entrada.

###Eliminación de registros duplicados.

In [ ]:
# Ver el número inicial de registros
print(f"Registros antes de quitar duplicados: {len(data)}")

# Eliminar duplicados exactos
data = data.drop_duplicates().reset_index(drop=True)

# Verificar cuántos quedaron
print(f"Registros después de quitar duplicados: {len(data)}")

### Tomo una muestra balanceada de 20.000 observaciones de (10.000 por clase)

In [ ]:
clase_0 = data[data['label'] == 0].sample(10000, random_state=42)
clase_1 = data[data['label'] == 1].sample(10000, random_state=42)

df_sample = pd.concat([clase_0, clase_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_sample['label'].value_counts())
print(f"Total observaciones: {len(df_sample)}")

In [ ]:
X_all= df_sample.drop(columns='label')
y_all= df_sample['label']

In [ ]:
print(f"Dimensión de la matriz de secuencias X: {X_all.shape}")
print(f"Dimensión del vector de etiquetas y: {y_all.shape}")

### **Separación en entrenamiento, test y validación.**

In [ ]:
#sSeparación 60% (train)/ 20%(test) /20% (val)

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42) # 0.25 x 0.8 = 0.2

# Imprimimos las dimensiones de los conjuntos
print(f"Dimensión del conjunto general de observaciones {X_all.shape}")
print(f"Dimensión del conjunto de entrenamiento {X_train.shape}")
print(f"Dimensión del conjunto de validación {X_val.shape}")
print(f"Dimensión del conjunto de prueba {X_test.shape}")

De lo anterior se observa que la división del conjunto de datos quedo así:

Entrenamiento: 12000 registros

Test: 4000 registros

Validación: 4000 registros


# **Limpieza de los datos**

In [ ]:
def clean_text(text):
    """
    Función que realiza la limpieza de texto eliminando etiquetas HTML,
    signos de puntuación, URLs y convirtiendo a minúsculas.
    """
    text = text.lower()                                            # Minúsculas
    text = re.sub(r'<br\s*/?>', ' ', text)                         # Etiquetas HTML
    text = re.sub(r'https?://\S+|www\.\S+', '', text)               # URLs
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)   # Signos de puntuación
    text = re.sub(r'\s+', ' ', text).strip()                       # Espacios redundantes
    return text

## **Pipeline de preprocesamiento de las reseñas**

Vamos a definir la función text_preprocess() para realizar todos los pasos del procesamiento de textos: limpieza, tokenización, eliminación de palabras vacías y stemming:

In [ ]:
# 1. Creamos la lista de stopwords pero excluyendo las negaciones importantes
stop_words = set(stopwords.words('english'))
negations = {'not', 'no', 'never', 'neither', 'nor', 'none'}
stop_words = stop_words - negations  # Mantenemos las negaciones vivas

def text_preprocess(text):
    
    text = clean_text(text)
    
    tokenizer = RegexpTokenizer(r'\w+')
    #tokenizer = RegexpTokenizer(r"\w+(?:['’]\w+)?")
    lemmatizer = WordNetLemmatizer()

    tokens = tokenizer.tokenize(text)

    # Filtramos usando el conjunto modificado que preserva las negaciones
    tokens = [word for word in tokens if word not in stop_words]

    # Aplicamos lematización
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(tokens)

In [ ]:
# Aplicar la función text_preprocess() a las reseñas
X_train_clean = [text_preprocess(text) for text in X_train['text']]
print(f"Dimensión entrenamiento de la muestra seleccionada: {len(X_train_clean)}")
print("Ejemplo de reseña original:")
print(X_train['text'].iloc[0][:200] + "...")
print("\nEjemplo de reseña limpia preprocesada:")
print(X_train_clean[0][:200] + "...")

X_val_clean = [text_preprocess(text) for text in X_val['text']]
X_test_clean = [text_preprocess(text) for text in X_test['text']]


### **Calculo de longitud promedio de cada reseña:**

In [ ]:
# Calcular la longitud de cada reseña limpia en palabras
lengths = [len(text.split()) for text in X_train_clean]

print(f"Longitud promedio: {np.mean(lengths):.1f} palabras")
print(f"Longitud máxima: {max(lengths)} palabras")
print(f"Percentil 90: {np.percentile(lengths, 90):.1f} palabras")
print(f"Percentil 95: {np.percentile(lengths, 95):.1f} palabras")
print(f"Percentil 99: {np.percentile(lengths, 99):.1f} palabras")

MAX_LEN = 250

**Con 250 palabras cubrimos más del 91% de las reseñas de forma completa sin tener que cortarles nada.**

In [ ]:
# Entrenar (o cargar) los Embeddings (Word2Vec) sobre el texto limpio
sentences_train = [text.split() for text in X_train_clean]
w2v_model = Word2Vec(
    sentences=sentences_train, vector_size=100, window=5, min_count=2
)

# Construir el vocabulario basado en los embeddings para hacer la conversión numérica
vocab = {word: idx + 2 for idx, word in enumerate(w2v_model.wv.index_to_key)}
vocab['<pad>'] = 0  # Índice para el relleno de secuencias
vocab['<unk>'] = 1  # Índice para palabras desconocidas


In [ ]:
# Función que convierte el texto limpio en la secuencia numérica requerida

def text_to_numeric_sequence(text, vocab, max_len=MAX_LEN):
  tokens = text.split()
  # Convierte cada palabra en su número/índice correspondiente del embedding
  sequence = [vocab.get(word, vocab['<unk>']) for word in tokens]

  # Asegurar longitud fija mediante Padding / Truncado
  if len(sequence) < max_len:
    sequence = sequence + [vocab['<pad>']] * (max_len - len(sequence))
  else:
    sequence = sequence[:max_len]
  return sequence

In [ ]:
# Aplicar el preprocesamiento final para obtener las secuencias numéricas de entrada
X_train_seq = torch.tensor(
    [text_to_numeric_sequence(t, vocab) for t in X_train_clean],
    dtype=torch.long,
)
X_val_seq = torch.tensor(
    [text_to_numeric_sequence(t, vocab) for t in X_val_clean], dtype=torch.long
)
X_test_seq = torch.tensor(
    [text_to_numeric_sequence(t, vocab) for t in X_test_clean], dtype=torch.long
)

**Descarga de vectores Glove pre-entrenados para el embedding:**

In [ ]:
target_file = "glove.6B.100d.txt"
zip_path = "glove.6B.zip"
url = "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip"
if not os.path.exists(target_file):
    # 1. Descargar el archivo zip si aún no está en la carpeta
    if not os.path.exists(zip_path):
        print("Descargando GloVe (~822 MB)... Esto puede tardar 2-4 minutos según tu conexión.")
        
        def show_progress(count, block_size, total_size):
            percent = count * block_size * 100 / total_size
            mb = count * block_size / (1024 * 1024)
            total_mb = total_size / (1024 * 1024)
            if count % 2000 == 0 or percent >= 100:
                print(f"\rDescargando: {percent:.1f}% ({mb:.1f} / {total_mb:.1f} MB)", end="")
        opener = urllib.request.build_opener()
        opener.addheaders = [('User-Agent', 'Mozilla/5.0')]
        urllib.request.install_opener(opener)
        urllib.request.urlretrieve(url, zip_path, reporthook=show_progress)
        print("\n¡Descarga completada!")
    # 2. Descomprimir el archivo glove.6B.100d.txt necesario para el modelo
    print("Descomprimiendo 'glove.6B.100d.txt'...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extract(target_file, ".")
    print("¡Listo! Archivo GloVe disponible para entrenar.")
else:
    print(f"El archivo '{target_file}' ya existe en el directorio.")

**Función para cargar e inyectar Embeddings pre-entrenados de GloVe**

In [ ]:
def load_glove_weights(glove_path, word_to_index, embedding_dim=100):
    vocab_size = len(word_to_index)
    # Inicializamos la matriz con ceros (o valores aleatorios pequeños)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    found_words = 0
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(' ')
            word = parts[0]
            if word in word_to_index:
                vector = np.array(parts[1:], dtype=float)
                index = word_to_index[word]
                embedding_matrix[index] = vector
                found_words += 1

    print(f"Palabras encontradas en GloVe: {found_words} de {vocab_size}")
    return torch.tensor(embedding_matrix, dtype=torch.float32)

## **Construcción arquitectura LSTM**

In [ ]:
class SentimentClassifierLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_units, output_dim=1, dropout=0.5):
        super().__init__()

        # 1. Capa de Embedding (específica para texto)
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0
        )

        # 2. LSTM Bidireccional
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_units,
            batch_first=True,
            bidirectional=True  # La red procesa la reseña de izquierda a derecha y de derecha a izquierda.
        )

        # 3. Capa de Dropout para regularización
        self.dropout = nn.Dropout(dropout)

        # 4. Capa Densa de salida (Multiplicada por 2 al ser bidireccional)
        self.fc = nn.Linear(hidden_units * 2, output_dim)

        # 5. Función de activación sigmoide
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: [batch_size, seq_len]

        # Pasamos por el embedding -> [batch, seq_len, embedding_dim]
        embedded = self.embedding(x)

        # Pasamos por la LSTM
        _, (h_n, _) = self.lstm(embedded)

        # Al ser bidireccional, h_n tiene 2 capas (forward y backward). Las concatenamos:
        # h_n shape: [num_layers * num_directions, batch, hidden_units]
        cat_hidden = torch.cat((h_n[-2, :, :], h_n[-1, :, :]), dim=1)

        # Aplicar el dropout al vector combinado antes de la capa lineal
        out = self.dropout(cat_hidden)

        # Capa lineal final y sigmoide para obtener la probabilidad binaria
        out = self.fc(out)
        return self.sigmoid(out).squeeze(1)

## **Instanciar el modelo.**


In [ ]:
def entrenar_y_evaluar(tipo_embedding='word2vec'):
    print("=" * 60)
    print(f" INICIANDO EXPERIMENTO: LSTM + {tipo_embedding.upper()} ")
    print("=" * 60)
        
    # 1. Definir los hiperparámetros
    VOCAB_SIZE = len(vocab)  # Tamaño total del diccionario (incluyendo <pad> y <unk>)
    EMBEDDING_DIM = 100  # Debe ser igual al vector_size de Word2Vec
    HIDDEN_UNITS = 64  # Número de neuronas ocultas en la LSTM
    OUTPUT_DIM = 1  # 1 sola salida (clasificación binaria: 0 o 1)
    DROPOUT = 0.5
    # Semilla para reproducibilidad
    seed_everything(42)

    # 2. Instanciar el modelo
    model = SentimentClassifierLSTM(
        vocab_size=VOCAB_SIZE,
        embedding_dim=EMBEDDING_DIM,
        hidden_units=HIDDEN_UNITS,
        output_dim=OUTPUT_DIM,
        dropout=DROPOUT,
    )

    # 3. Asignar Embeddings correspondientes
    if tipo_embedding == 'word2vec':
        # Extraemos los vectores directamente del modelo Word2Vec entrenado
        embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))
        for word, idx in vocab.items():
            if word in w2v_model.wv:
                embedding_matrix[idx] = w2v_model.wv[word]

        model.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))

    elif tipo_embedding == 'glove':
        # Invocar función para cargar y asignar pesos de GloVe usando el mismo vocabulario
        glove_weights = load_glove_weights('glove.6B.100d.txt', vocab, EMBEDDING_DIM)
        model.embedding.weight.data.copy_(glove_weights)
    
    model.embedding.weight.requires_grad = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # 4. DataLoaders
    train_loader = DataLoader(TensorDataset(X_train_seq, torch.tensor(y_train.values, dtype=torch.float32)), batch_size=128, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_seq, torch.tensor(y_val.values, dtype=torch.float32)), batch_size=128, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_test_seq, torch.tensor(y_test.values, dtype=torch.float32)), batch_size=128, shuffle=False)

    #**Definición de función de pérdida y optimizador:**

    # nn.BCELoss: Entropía cruzada binaria, ideal para problemas de clasificación binaria
    # (equivalente a binary_crossentropy); requiere que la capa de salida ya entregue probabilidades
    #  (por ejemplo, tras una Sigmoid).

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    
    # 5. Bucle de Entrenamiento
    n_epochs, patience = 30, 3
    best_val_loss, best_state = float('inf'), None
    epochs_no_improve = 0
    history = {'loss': [], 'val_loss': []}
    start_time = time()
    
    for epoch in range(n_epochs):
        model.train()
        r_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            r_loss += loss.item() * xb.size(0)
            
        t_loss = r_loss / len(train_loader.dataset)
        
        # Validación
        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for v_xb, v_yb in val_loader:
                v_xb, v_yb = v_xb.to(device), v_yb.to(device)
                v_loss += criterion(model(v_xb), v_yb).item() * v_xb.size(0)
        v_loss = v_loss / len(val_loader.dataset)
        
        history['loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"Época {epoch+1:02d} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}")
            
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            epochs_no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"-> Parada temprana en la época {epoch+1}")
                break
                
    model.load_state_dict(best_state)
    elapsed = time() - start_time
    
    # 6. Evaluación en Prueba
    model.eval()
    preds_all, y_all = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds_all.append(model(xb.to(device)).cpu())
            y_all.append(yb)
            
    y_pred = (torch.cat(preds_all) >= 0.5).int().numpy()
    y_true = torch.cat(y_all).int().numpy()
    
    metricas = {
        'Modelo': f"LSTM + {tipo_embedding.upper()}",
        'Accuracy (%)': accuracy_score(y_true, y_pred) * 100,
        'Precision': precision_score(y_true, y_pred, average='binary'),
        'Recall': recall_score(y_true, y_pred, average='binary'),
        'F1-Score': f1_score(y_true, y_pred, average='binary'),
        'Tiempo (s)': elapsed
    }
    
    return metricas, history, y_true ,  y_pred

In [ ]:
# 1. Ejecutar las dos opciones
resultados = []

res_w2v, hist_w2v,  y_truew2v ,  y_predw2v = entrenar_y_evaluar('word2vec')
resultados.append(res_w2v)

res_glove, hist_glove, y_true_g, y_pred_g = entrenar_y_evaluar('glove')
resultados.append(res_glove)

# 2. Imprimir Tabla Comparativa
df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print("\n" + "=" * 55)
print(" TABLA COMPARATIVA DE RESULTADOS FINALES")
print("=" * 55)
display(df_resultados.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# Curvas Word2Vec
axes[0].plot(hist_w2v['loss'], label='Pérdida Entrenamiento', color='#5a2ff4', lw=2)
axes[0].plot(hist_w2v['val_loss'], label='Pérdida Validación', color='#60d13e', lw=2)
axes[0].set_title('LSTM + Word2Vec (Local)', fontsize=12)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida (BCE)')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# Curvas GloVe
axes[1].plot(hist_glove['loss'], label='Pérdida Entrenamiento', color='#2b5c8f', lw=2)
axes[1].plot(hist_glove['val_loss'], label='Pérdida Validación', color='#e67e22', lw=2)
axes[1].set_title('LSTM + GloVe (Pre-entrenado)', fontsize=12)
axes[1].set_xlabel('Época')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# Visualización de la Matriz de Confusión
# ========================================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# --- FILA 1: Conteos Absolutos ---
# Word2Vec
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_truew2v, y_predw2v),
    display_labels=['Negativa (0)', 'Positiva (1)']
).plot(ax=axes[0, 0], cmap='Purples', values_format='d')
axes[0, 0].set_title('LSTM + Word2Vec (Conteos Absolutos)', fontsize=12, pad=8)
axes[0, 0].grid(False)
# GloVe
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_true_g, y_pred_g),
    display_labels=['Negativa (0)', 'Positiva (1)']
).plot(ax=axes[0, 1], cmap='Blues', values_format='d')
axes[0, 1].set_title('LSTM + GloVe (Conteos Absolutos)', fontsize=12, pad=8)
axes[0, 1].grid(False)
# --- FILA 2: Normalizadas por Clase Real (%) ---
# Word2Vec
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_truew2v, y_predw2v, normalize='true'),
    display_labels=['Negativa (0)', 'Positiva (1)']
).plot(ax=axes[1, 0], cmap='Purples', values_format='.2%')
axes[1, 0].set_title('LSTM + Word2Vec (Normalizada %)', fontsize=12, pad=8)
axes[1, 0].grid(False)
# GloVe
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_true_g, y_pred_g, normalize='true'),
    display_labels=['Negativa (0)', 'Positiva (1)']
).plot(ax=axes[1, 1], cmap='Blues', values_format='.2%')
axes[1, 1].set_title('LSTM + GloVe (Normalizada %)', fontsize=12, pad=8)
axes[1, 1].grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# Análisis Cualitativo de Predicciones: Aciertos y Errores
# ========================================================
import pandas as pd

# Variables de etiquetas reales y predicciones
y_true_real = y_truew2v  # Mismo conjunto de prueba
y_pred_w = y_predw2v

label_map = {0: "Negativa (0)", 1: "Positiva (1)"}

# 1. Construir DataFrame con el texto original y las predicciones de ambos modelos
df_cualitativo = pd.DataFrame({
    "texto_original": X_test["text"].values,
    "etiqueta_real": y_true_real,
    "pred_w2v": y_pred_w,
    "pred_glove": y_pred_g,
    "acierto_w2v": (y_true_real == y_pred_w),
    "acierto_glove": (y_true_real == y_pred_g)
})

# 2. Clasificar cada predicción en una categoría de análisis
def clasificar_resultado(row):
    if row["acierto_w2v"] and row["acierto_glove"]:
        return "Acierto Ambos"
    elif not row["acierto_w2v"] and not row["acierto_glove"]:
        return "Fallo Ambos (Casos Difíciles)"
    elif row["acierto_glove"] and not row["acierto_w2v"]:
        return "Acierta GloVe / Falla Word2Vec"
    else:
        return "Acierta Word2Vec / Falla GloVe"

df_cualitativo["categoria"] = df_cualitativo.apply(clasificar_resultado, axis=1)

# 3. Resumen estadístico de concordancia/discordancia
print("=" * 65)
print(" DISTRIBUCIÓN CUALITATIVA DE PREDICCIONES (TEST SET)")
print("=" * 65)
resumen = df_cualitativo["categoria"].value_counts().to_frame(name="Total Muestras")
resumen["Porcentaje (%)"] = (resumen["Total Muestras"] / len(df_cualitativo)) * 100
display(resumen.round(2))

# 4. Función para inspeccionar ejemplos concretos
def inspeccionar_caso(categoria_nombre, titulo, n=1, seed=42):
    sub = df_cualitativo[df_cualitativo["categoria"] == categoria_nombre]
    if sub.empty:
        print(f"No hay casos en la categoría: {categoria_nombre}")
        return
    
    muestra = sub.sample(n=min(n, len(sub)), random_state=seed)
    print("\n" + "=" * 80)
    print(f" {titulo.upper()}")
    print("=" * 80)
    
    for idx, row in muestra.iterrows():
        estado_w2v = "✅ ACIERTO" if row['acierto_w2v'] else "❌ ERROR"
        estado_glove = "✅ ACIERTO" if row['acierto_glove'] else "❌ ERROR"
        
        print(f"\n[Muestra #{idx}]")
        print(f"  • Sentimiento Real : {label_map[row['etiqueta_real']]}")
        print(f"  • Pred. Word2Vec   : {label_map[row['pred_w2v']]}  --> {estado_w2v}")
        print(f"  • Pred. GloVe      : {label_map[row['pred_glove']]}  --> {estado_glove}")
        print(f"  • Fragmento reseña :")
        
        texto_limpio = row['texto_original'].replace('\n', ' ').strip()
        print(f"    \"{texto_limpio[:320]}...\"")

# 5. Imprimir casos representativos para el análisis cualitativo
inspeccionar_caso("Acierto Ambos", "1. Acierto Mutuo (Sentimiento directo y evidente)", n=1, seed=10)
inspeccionar_caso("Acierta GloVe / Falla Word2Vec", "2. GloVe acierta donde Word2Vec falla (Vocabulario general más rico)", n=1, seed=20)
inspeccionar_caso("Acierta Word2Vec / Falla GloVe", "3. Word2Vec acierta donde GloVe falla (Adaptación al dominio cinematográfico)", n=1, seed=30)
inspeccionar_caso("Fallo Ambos (Casos Difíciles)", "4. Falla Mutua (Sarcasmo, ironía o ambivalencia)", n=1, seed=40)